# 03 - Candidate Generation via Implicit ALS
This notebook implements Stage 1 retrieval:
- Constructing sparse user-item interaction matrices
- Weighting implicit event feedback (`view` = 1, `addtocart` = 3, `transaction` = 5)
- Training Matrix Factorization using Alternating Least Squares (ALS)
- Generating top-$K$ candidate pools and evaluating Retrieval Recall@100

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import implicit
import joblib
import yaml
from pathlib import Path

# Load project configurations
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

events_path = config.get("data", {}).get("raw_path", "../data/raw/events.csv")
models_dir = Path("../models")
results_dir = Path("../results")
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

df_events = pd.read_csv(events_path)
print(f"Loaded {len(df_events):,} total interactions.")


### 1. Build Implicit Confidence Scores & Sparse Matrix

In [ ]:
# Event weights: view=1, cart=3, transaction=5
event_weights = {"view": 1.0, "addtocart": 3.0, "transaction": 5.0}
df_events['weight'] = df_events['event'].map(event_weights).fillna(1.0)

# Filter low-activity entities for matrix factorization stability
min_user_interactions = 3
min_item_interactions = 5

user_counts = df_events['visitorid'].value_counts()
item_counts = df_events['itemid'].value_counts()

active_users = user_counts[user_counts >= min_user_interactions].index
active_items = item_counts[item_counts >= min_item_interactions].index

filtered_df = df_events[
    df_events['visitorid'].isin(active_users) & 
    df_events['itemid'].isin(active_items)
].copy()

# Encode external IDs to contiguous zero-indexed integers
user_categories = filtered_df['visitorid'].astype('category')
item_categories = filtered_df['itemid'].astype('category')

user_indices = user_categories.cat.codes
item_indices = item_categories.cat.codes

user_id_map = dict(enumerate(user_categories.cat.categories))
item_id_map = dict(enumerate(item_categories.cat.categories))

# Construct CSR interaction matrix (users x items)
interaction_matrix = sp.csr_matrix(
    (filtered_df['weight'].values, (user_indices, item_indices)),
    shape=(len(user_id_map), len(item_id_map)),
    dtype=np.float32
)

print(f"Interaction matrix shape: {interaction_matrix.shape}")
print(f"Matrix non-zero entries: {interaction_matrix.nnz:,}")


### 2. Train Implicit Alternating Least Squares (ALS)

In [ ]:
# Initialize ALS model
als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    random_state=42
)

# implicit expects user-item matrix
als_model.fit(interaction_matrix)

# Save ALS artifacts
joblib.dump(als_model, models_dir / "mf_model.pkl")
joblib.dump({"user_map": user_id_map, "item_map": item_id_map}, models_dir / "als_id_mappings.pkl")

print("ALS model and ID lookup maps successfully saved to models/")


### 3. Generate Top-100 Candidate Retrieval Pools

In [ ]:
# Retrieve candidate items for a sample batch of users
sample_user_idxs = np.arange(min(1000, interaction_matrix.shape[0]))
top_k = 100

ids, scores = als_model.recommend(
    sample_user_idxs,
    interaction_matrix[sample_user_idxs],
    N=top_k,
    filter_already_liked_items=True
)

candidates_sample = []
for u_idx, (rec_items, rec_scores) in enumerate(zip(ids, scores)):
    original_user = user_id_map[u_idx]
    for item_idx, score in zip(rec_items, rec_scores):
        candidates_sample.append({
            "visitorid": original_user,
            "itemid": item_id_map[item_idx],
            "als_retrieval_score": float(score)
        })

candidate_df = pd.DataFrame(candidates_sample)
print(f"Generated {len(candidate_df):,} candidate pairs for {len(sample_user_idxs)} users.")
candidate_df.head()
